# Data Collection and Feature Engineering

This notebook pulls daily price data for 14 tickers across six sectors, builds a small set of technical and macro features, and constructs the next-day label columns. Of these, only the volatility spike label carries into the model comparison, with the direction columns kept as an EDA baseline. Everything is set up so a row at time t only uses information available at t, and every target describes t+1.

It writes two files, a full dataset that keeps early rows with incomplete lookback windows for EDA, and a clean dataset with complete features for modeling.

In [ ]:
# pip install -r requirements.txt

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ta
from datetime import date, timedelta
from pathlib import Path

# Dynamic timeline (end is exclusive, so add a day to grab latest close)
start_date = date(2015, 1, 1)
end_date = date.today() + timedelta(days=1)


In [ ]:
# Paths and simple existence checks
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

raw_path = ROOT / "data" / "raw" / "merged_features_full.csv"
clean_path = ROOT / "data" / "processed" / "merged_features_clean.csv"

print(f"Raw exists: {raw_path.exists()} -> {raw_path.relative_to(ROOT)}")
print(f"Clean exists: {clean_path.exists()} -> {clean_path.relative_to(ROOT)}")


In [ ]:
# Cross-sector ticker mix so results are not tech-only
tickers = ["AAPL","MSFT","AMZN","GOOGL","NVDA","TSLA","JPM","WMT",
           "DAL","UAL","LMT","RTX","NOC","XOM"]

print(f"{len(tickers)} tickers")

In [ ]:
data = yf.download(tickers, start=start_date, end=end_date, auto_adjust=True)

In [ ]:
# Flatten multi-index into one row per ticker and date
data_flat = data.stack(level=1, future_stack=True).reset_index()

data_flat.rename(columns={
    "level_1": "Ticker",
    "Adj Close": "AdjClose",
    "Close": "Close",
    "Open": "Open",
    "High": "High",
    "Low": "Low",
    "Volume": "Volume"
}, inplace=True)

data_flat.head()

## Features

The feature set is intentionally small and interpretable rather than a large collection of indicators.
- **Return** and **RollingVol** capture recent movement and its 10-day volatility
- **Lagged_Return** is the prior day's return, engineered alongside the targets below
- **RSI** is a standard momentum gauge
- **Price/SMA ratios** are used instead of raw moving averages so values are comparable across tickers and don't leak price scale
- **Volume_Z** standardizes volume within each ticker
- **VIX** brings in market-wide stress as a macro signal

In [ ]:
# Feature engineering: technical indicators
# Sort chronologically within each ticker so rolling windows are well-defined
data_flat = data_flat.sort_values(["Ticker", "Date"]).reset_index(drop=True)

data_flat["Return"] = data_flat.groupby("Ticker")["Close"].pct_change()

data_flat["RollingVol"] = (
    data_flat.groupby("Ticker")["Return"]
    .rolling(window=10)
    .std()
    .reset_index(0, drop=True)
)

data_flat["RSI"] = data_flat.groupby("Ticker")["Close"].transform(
    lambda x: ta.momentum.rsi(x, window=14)
)

# SMA ratios instead of absolute levels to avoid price scale leakage
sma_20 = data_flat.groupby("Ticker")["Close"].transform(
    lambda x: x.rolling(20).mean()
)
sma_50 = data_flat.groupby("Ticker")["Close"].transform(
    lambda x: x.rolling(50).mean()
)

data_flat["Price_to_SMA20"] = data_flat["Close"] / sma_20
data_flat["Price_to_SMA50"] = data_flat["Close"] / sma_50
data_flat["SMA20_to_SMA50"] = sma_20 / sma_50

data_flat["Volume_Z"] = data_flat.groupby("Ticker")["Volume"].transform(
    lambda x: (x - x.rolling(20).mean()) / x.rolling(20).std()
)

In [ ]:
# Merge VIX index (leakage-safe forward-fill)
macro = yf.download(
    ["^VIX"],
    start=start_date,
    end=end_date,
    auto_adjust=True
)

macro.columns = [col[0] for col in macro.columns]
macro = macro.reset_index().rename(columns={"Close": "VIX"})

# forward-fill VIX before the merge to avoid cross-ticker leakage
macro = macro.sort_values("Date")
macro["VIX"] = macro["VIX"].ffill()

data_flat = data_flat.merge(macro[["Date", "VIX"]], on="Date", how="left")

print(f"Leading VIX NaNs: {data_flat['VIX'].isna().sum()}")

## Targets

All targets look one day ahead. NextVolSpike is the next-day demonstration target and flags whether tomorrow's rolling volatility exceeds its own expanding 80th percentile, using only history up to t-1, with today's value excluded via `shift(1)`, so there is no look-ahead. NextReturn and NextDirection are built as well, but only as a direction baseline for the volatility-versus-direction comparison in the EDA.

The modeling notebook later shows this next-day spike target mostly restates today's regime by construction (the 10-day windows at t and t+1 overlap), so a forward-disjoint target over the next 10 days is derived in `src/targets.py` and used for the main results.

In [ ]:
# Re-sort after the VIX merge so the per-ticker shifts below run in date order
data_flat = data_flat.sort_values(["Ticker", "Date"]).reset_index(drop=True)

# Lagged_Return: prior day's return (t-1) for predicting t+1 outcomes
data_flat["Lagged_Return"] = data_flat.groupby("Ticker")["Return"].shift(1)


# Create targets (t → t+1 labels, leakage-safe)

data_flat["NextReturn"] = data_flat.groupby("Ticker")["Return"].shift(-1)

data_flat["NextDirection"] = np.where(
    data_flat["NextReturn"].isna(),
    np.nan,
    (data_flat["NextReturn"] > 0).astype(float)
)

# NextVolSpike(t) = 1 if RollingVol at t+1 exceeds 80th percentile of
# historical RollingVol computed using only data up to t-1

next_day_vol = data_flat.groupby("Ticker")["RollingVol"].shift(-1)

# Expanding threshold uses only information available up to t-1
# shift(1) keeps today's RollingVol out of the threshold
expanding_threshold = data_flat.groupby("Ticker")["RollingVol"].transform(
    lambda s: s.shift(1).expanding(min_periods=126).quantile(0.8)
)

data_flat["NextVolSpike"] = np.where(
    next_day_vol.isna() | expanding_threshold.isna(),
    np.nan,
    (next_day_vol > expanding_threshold).astype(float)
)

# Persist the label-defining threshold so downstream code (the matched naive baseline and
# FwdVolRegime in src/targets.py) reuses it instead of re-deriving it from the
# warm-up-trimmed clean file, where every expanding window would start late.
data_flat["vol_threshold"] = expanding_threshold

# Verify per-target NaN counts match the expected shift and warm-up losses
print("Target NaN counts (from shifting/warm-up windows):")
print(f"  Lagged_Return: {data_flat['Lagged_Return'].isna().sum()} NaNs")
print(f"  NextReturn:    {data_flat['NextReturn'].isna().sum()} NaNs")
print(f"  NextDirection: {data_flat['NextDirection'].isna().sum()} NaNs")
print(f"  NextVolSpike:  {data_flat['NextVolSpike'].isna().sum()} NaNs")
print(f"\nExpected NaNs:")
print(f"  Lagged_Return: 14 (1 per ticker from shift)")
print(f"  NextReturn & NextDirection: 14 (1 per ticker from shift)")
print(f"  NextVolSpike: ~1,890 (126-day warm-up + 1 shift per ticker)")

In [ ]:
raw_path.parent.mkdir(parents=True, exist_ok=True)
clean_path.parent.mkdir(parents=True, exist_ok=True)


In [ ]:
# The full file keeps incomplete-feature rows for EDA price history
data_flat.to_csv(raw_path, index=False)

# Drop rows after all feature/label engineering to remove terminal rows
data_flat_clean = data_flat.dropna().copy()

# int cast is safe once dropna has removed the NaN labels
data_flat_clean["NextDirection"] = (
    data_flat_clean["NextDirection"].astype(int)
)
data_flat_clean["NextVolSpike"] = data_flat_clean["NextVolSpike"].astype(int)

data_flat_clean.to_csv(clean_path, index=False)

print(f"Full dataset (with incomplete features): {data_flat.shape[0]} rows")
print(f"Clean dataset (model-ready): {data_flat_clean.shape[0]} rows")
print(f"Rows dropped (incomplete features + NaN targets): "
      f"{data_flat.shape[0] - data_flat_clean.shape[0]}")
print(f"\nClean dataset target stats:")
print(f"  NextDirection: "
      f"{data_flat_clean['NextDirection'].value_counts().to_dict()}")
print(f"  NextVolSpike:  "
      f"{data_flat_clean['NextVolSpike'].value_counts().to_dict()}")

# a shift bug would leave Lagged_Return identical to Return
print(f"\nLagged_Return validation:")
print(f"  Correlation(Return, Lagged_Return): "
      f"{data_flat_clean['Return'].corr(data_flat_clean['Lagged_Return']):.6f}")
print(f"  Are they identical? "
      f"{(data_flat_clean['Return'] == data_flat_clean['Lagged_Return']).all()}")

data_flat_clean.head(10)

### Dataset outputs

Two files are written.
- `merged_features_full.csv` (~38,920 rows from Jan 2015, keeps rows with incomplete features so EDA can see the whole price history)
- `merged_features_clean.csv` (~37,002 rows from 2015-07-20 onward, where every indicator has a complete lookback window, used for modeling).

The notebook also persists `vol_threshold`, the expanding 80th-percentile threshold that defines the volatility labels, into both CSVs. `scripts/verify_vol_threshold.py` re-derives it from the raw file and asserts it reproduces the stored `NextVolSpike` labels exactly before writing.